# url4 Server — standing up a node

`Url4Node` is the node-side SDK: register **intent processors** (endpoints),
**holdings** (`@` / `@identity`), and **data routes** with decorators, then evaluate
expressions in-process — or serve the exact same dispatch over HTTP GET.

A node **is** an IOLayer: in-process evaluation, engine-internal sub-requests, and
the HTTP surface all flow through one `fetch()`, so they can never diverge.

In [1]:
from url4 import Request, ResolutionError, StaticIOLayer, Url4Error, Url4Node

# the outside world this node can reach (offline stand-in for real HTTP)
world = StaticIOLayer(
    fetch_map={
        "https://news.example.com/ai-act": (
            "EU lawmakers approved the AI Act: risk tiers, transparency duties, "
            "and human oversight for high-risk uses; rules phase in over two years."
        ),
    }
)

node = Url4Node("library.example.ai", outbound=world)
node

<Url4Node 'library.example.ai': empty>

## Endpoints — intent processors

An endpoint handler receives a `Request` with the decoded `(context, intent)`
pair. A real node would call an LLM gateway here; `context` arrives exactly as
the wire carries it (an engine reducer dispatch packs **resolved** data into the
intent; a relative-expression call passes its inner context through for this
node to work on).

In [2]:
WIRE: list[Request] = []  # capture what handlers receive, for inspection


def _echo(model: str, request: Request) -> str:
    WIRE.append(request)
    topic = request.context.strip().splitlines()[0][:48] if request.context.strip() else ""
    tail = f" — re: {topic!r}" if topic else ""
    return f"[{model}] {request.intent.splitlines()[0][:60]}{tail}"


@node.endpoint("/claude")
async def claude(request: Request) -> str:
    return _echo("claude", request)


@node.endpoint("/gemini")
def gemini(request: Request) -> str:  # sync handlers work too
    return _echo("gemini", request)

## Evaluate in-process

`node.evaluate()` runs any url4 expression with this node as its whole world:
relative expressions dispatch to the endpoints above, absolute URLs go outbound,
and a fan-out's reducer goes to the node's `default_processor` (`/claude`).

In [3]:
res = await node.evaluate(
    "(a:0.6:/claude(https://news.example.com/ai-act)!'Summarize',"
    " b:0.4:/gemini(https://news.example.com/ai-act)!'Summarize')"
    "!'Merge $a and $b'"
)
print(res.text)

[claude] a (weight=0.6):


In [4]:
last = WIRE[-1]  # the reducer dispatch: resolved fan-out results packed as the intent
print("path:   ", last.path)
print("context:", repr(last.context))
print("intent: ", last.intent[:160].replace(chr(10), " | "))

path:    /claude
context: ''
intent:  a (weight=0.6): | [claude] Summarize — re: 'https://news.example.com/ai-act' |  | b (weight=0.4): | [gemini] Summarize — re: 'https://news.example.com/ai-act' |  | [Instruc


## Holdings — `@` is the node's own data

`@` resolves against decorator-registered holdings (spec §5.6.1); a collection
qualifier selects a shelf.

In [5]:
@node.holdings  # bare decorator = the default shelf; handlers may take no args
def default_holdings() -> str:
    return "General stacks: 12,000 documents on AI governance and policy."


@node.holdings("science")
def science_holdings() -> str:
    return "Science shelf: 4,000 peer-reviewed ML papers."


print((await node.evaluate("(@)!'What do you hold?'")).text)

# NOTE: '@name' always means a PRINCIPAL (next section) — the spec selects a
# self-collection via the endpoint's URL path (§5.6.3.1), an engine follow-up.
# The holdings port itself already serves shelves:
print(await node.fetch_holdings(None, "science"))

What do you hold?

General stacks: 12,000 documents on AI governance and policy.
Science shelf: 4,000 peer-reviewed ML papers.


## Identities — `@name` is someone else's data

An identity handler resolves a principal's holdings and can gate access with the
spec's error codes (`identity_access_denied`, `consent_required`, …). Unknown
principals fail permanently with `unknown_identity` (spec §5.6.3.2).

In [6]:
@node.identity("emily")
def emily(collection: str | None) -> str:
    if collection == "private":
        raise ResolutionError(
            "emily has not consented to this use",
            code="identity_access_denied",
            permanent=True,
        )
    return f"Emily's {collection or 'public'} notes: cautiously optimistic on AI."


print((await node.evaluate("(@emily/notes)!'Summarize her stance'")).text)

for bad in ("(@emily/private)!'peek'", "(@bob)!'anything'"):
    try:
        await node.evaluate(bad)
    except Url4Error as exc:
        print(f"{bad:28s} -> {exc.code}: {exc}")

Summarize her stance

Emily's notes notes: cautiously optimistic on AI.
(@emily/private)!'peek'      -> identity_access_denied: emily has not consented to this use
(@bob)!'anything'            -> unknown_identity: unknown identity 'bob' on node 'library.example.ai'


## Data routes — plain reads the engine can iterate

`node.data()` serves source material (`/api/…` reads) — a literal value or a
decorated provider — which makes it a collection for the `*` operator.

In [7]:
node.data("/api/tickets", '[{"q": "How do I reset my password?"}, {"q": "Can I export my data?"}]')


@node.data("/api/motd")  # or decorate a provider function
def motd() -> str:
    return "The reading room closes at midnight."


res = await node.evaluate("/api/tickets*(/claude($item.q)!'Answer')")
res.elements

["[claude] Answer — re: 'How do I reset my password?'",
 "[claude] Answer — re: 'Can I export my data?'"]

## The eval path — the protocol surface

`GET /v1?[params&]q=<expression>` is where external requestors land: the node
evaluates the full expression (sources resolve *here*), re-attaching protocol
params so `broadcast` / `quorum` keep their spec meaning (§6.1.1, §9).

In [8]:
print(await node.fetch("/v1?q=(@)!'inventory, please'", relative=True))
print(await node.fetch("/v1?broadcast&q=('AI Act', 'EU funding')!'Headline'", relative=True))

inventory, please

General stacks: 12,000 documents on AI governance and policy.
[{"source_position": 1, "source_name": null, "result": "Headline\n\nAI Act"}, {"source_position": 2, "source_name": null, "result": "Headline\n\nEU funding"}]


## Serving over HTTP — the ASGI shim

`node.asgi()` is a plain, framework-free ASGI app around the same dispatch —
GET-only (the url4 expression *is* the address, so the transactional verb is an
idempotent, cacheable GET). Driven here through `httpx.ASGITransport`; in
production, `node.serve(port=4404)` runs it with uvicorn (`pip install url4[server]`).

In [9]:
import httpx

http = httpx.AsyncClient(
    transport=httpx.ASGITransport(app=node.asgi()),
    base_url="http://library.example.ai",
)

r = await http.get("/v1", params={"q": "(@)!'What do you hold?'"})
r.status_code, r.text

(200,
 'What do you hold?\n\nGeneral stacks: 12,000 documents on AI governance and policy.')

In [10]:
# spec error codes map onto HTTP statuses
for path, q in [("/v1", "(@bob)!'x'"), ("/v1", "(@emily/private)!'x'"), ("/nope", "()!'x'")]:
    r = await http.get(path, params={"q": q})
    print(r.status_code, r.json()["error"]["code"])

print((await http.post("/v1", content=b"")).status_code, "<- POST (url4 speaks GET)")

404 unknown_identity
403 identity_access_denied
404 endpoint_not_found
405 <- POST (url4 speaks GET)


## Closing the loop — a `Client` against this node

The requestor side from `url4_client.ipynb`, pointed at this node's eval path over
(simulated) HTTP: the client renders the canonical remote URI, the node resolves
the sources, and the intent lands on the `/claude` processor.

In [11]:
from url4 import Client, HttpIOLayer, src

# over real HTTP this is just: Client("url4://library.example.ai/v1")
client = Client(HttpIOLayer(client=http), node="url4://library.example.ai/v1")
res = await client.query(
    src("https://news.example.com/ai-act", name="article"),
    intent="Summarize $article",
)
print("request:", res.request)
print("result: ", res.text)

request: (url4://library.example.ai/v1(article=https://news.example.com/ai-act)!'Summarize $article')
result:  Summarize EU lawmakers approved the AI Act: risk tiers, transparency duties, and human oversight for high-risk uses; rules phase in over two years.


---
### What a production node adds from here

- `node.serve(host, port)` — uvicorn serving (the `url4[server]` extra).
- Real endpoints calling an AI gateway; real `outbound` (drop the argument and
  the node owns an httpx adapter).
- Deferred by design until the transport spec lands: response envelopes,
  streaming delivery, requestor authentication / consent enforcement.